In [67]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler


In [83]:
#merging files

impaye = pd.read_table("Impayé dev.txt")
mej = pd.read_table("MEJ dev.txt")
ind_fin = pd.read_table('Ind financiers dev.txt',encoding = 'cp1252')
prod = pd.read_table("prod dev.txt",encoding='cp1252')

base = prod.copy()

impaye = impaye[["IDENTIFIANT_CREDIT","NBRE_JOUR_IMPAYE"]]
impaye = impaye.groupby("IDENTIFIANT_CREDIT").sum().reset_index()
mej = mej[["IDENTIFIANT_CREDIT","MONTANT_DEMANDE_MEJ"]]
mej = mej.drop_duplicates(subset=['IDENTIFIANT_CREDIT'])
defaut = pd.merge(impaye, mej, on="IDENTIFIANT_CREDIT", how="outer")

prod_def = pd.merge(prod,defaut,on="IDENTIFIANT_CREDIT",how="outer")

ind_fin["DATEEXER"] = pd.to_datetime(ind_fin["DATEEXER"])
ind_fin["VALEUR_FINALE"] = pd.to_numeric(ind_fin["VALEUR_FINALE"].str.replace(",","."))

ind_fin = ind_fin[ind_fin["DATEEXER"].dt.year <= 2019]

ind_fin["IDENTIFIANT_TIERS"] = ind_fin["IDENTIFIANT_TIERS"].astype(str).str.strip()
prod["IDENTIFIANT_TIERS"] = prod["IDENTIFIANT_TIERS"].astype(str).str.strip()

ind_wide = ind_fin.pivot_table(
    index='IDENTIFIANT_TIERS',
    columns = "CODE_INDICATEUR",
    values = "VALEUR_FINALE"
)

df = pd.merge(prod_def,ind_wide,on = "IDENTIFIANT_TIERS",how = "outer")

cond_impaye = df["NBRE_JOUR_IMPAYE"].notna() 

cond_mej = df["MONTANT_DEMANDE_MEJ"].notna()

df["DEFAUT"] = (cond_impaye | cond_mej).astype(int)

data = df.copy()

/var/folders/dv/rdq6c5q146v6p2q8m1nyh6q00000gn/T/ipykernel_90105/1435289583.py:5: DtypeWarning: Columns (0: IDENTIFIANT_TIERS, 1: IDENTIFIANT_TIERS_1) have mixed types. Specify dtype option on import or set low_memory=False.
  ind_fin = pd.read_table('Ind financiers dev.txt',encoding = 'cp1252')
/var/folders/dv/rdq6c5q146v6p2q8m1nyh6q00000gn/T/ipykernel_90105/1435289583.py:18: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  ind_fin["DATEEXER"] = pd.to_datetime(ind_fin["DATEEXER"])


In [69]:
#save si besoin
df.to_csv("dataset_risque_credit.csv")

In [70]:
cols_to_drop = ["NBRE_JOUR_IMPAYE", "MONTANT_DEMANDE_MEJ", "CLASSE_RISQUE", "FINALITE_CREDIT",'IDENTIFIANT_TIERS', 'IDENTIFIANT_CREDIT']

cols_present = [col for col in cols_to_drop if col in df.columns]
cols_missing = [col for col in cols_to_drop if col not in df.columns]

df = df.drop(columns=cols_present)


In [71]:
date_cols = ['DATE_CREATION_ENTREPRISE', 'DATE_ACCORD', 'DATE_PREMIER_DEBLOCAGE']

cols_present = [col for col in date_cols if col in df.columns]
cols_missing = [col for col in date_cols if col not in df.columns]

for col in cols_present:
    df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='raise')


try : 
    df['AGE_ENTREPRISE_JOURS'] = (df['DATE_ACCORD'] - df['DATE_CREATION_ENTREPRISE']).dt.days
    df['AGE_ENTREPRISE_ANS'] = df['AGE_ENTREPRISE_JOURS'] / 365.25
except : 
    pass


df = df.drop('AGE_ENTREPRISE_JOURS',axis =1)
df = df.drop(columns=cols_present)


In [72]:
raw_cols = ['DUREE_CONCOURS', 'UNITE_DUREE_CONCOURS', 'PERIODICITE']
cols_present = [col for col in raw_cols if col in df.columns]

unite_to_mois = {'mois': 1, 'an(s)': 12}
periodicite_mois = {
    'Mensuelle': 1, 'Trimestrielle': 3, 'Quadrimestrielle': 4,
    'Semestrielle': 6, 'Annuelle': 12
}

if 'DUREE_CONCOURS_MOIS' not in df.columns:
    df['DUREE_CONCOURS_MOIS'] = df['DUREE_CONCOURS'] * df['UNITE_DUREE_CONCOURS'].map(unite_to_mois)

if 'PERIODICITE_MOIS' not in df.columns:
    df['PERIODICITE_MOIS'] = df['PERIODICITE'].map(periodicite_mois)

if 'FLAG_PERIODICITE_MANQUANTE' not in df.columns:
    df['FLAG_PERIODICITE_MANQUANTE'] = df['PERIODICITE'].isna().astype(int)

if 'NBRE_ECHEANCES' not in df.columns:
    df['NBRE_ECHEANCES'] = df['DUREE_CONCOURS_MOIS'] / df['PERIODICITE_MOIS']

df['DUREE_CONCOURS_MOIS'] = df['DUREE_CONCOURS_MOIS'].fillna(df['DUREE_CONCOURS_MOIS'].median())
df['NBRE_ECHEANCES'] = df['NBRE_ECHEANCES'].fillna(df['NBRE_ECHEANCES'].median())

df = df.drop(columns=cols_present)

In [73]:
low_card_cols = ['REGION_RC', 'REGION_TIERS', 'CENTRE_AFFAIRE']
high_card_cols = ['VILLE_RC', 'VILLE_TIERS']

low_card_present = [c for c in low_card_cols if c in df.columns]
high_card_present = [c for c in high_card_cols if c in df.columns]

df = pd.get_dummies(df, columns=low_card_present, drop_first=True)

for col in high_card_present:
    freq_map = df[col].value_counts(normalize=True)
    df[col + '_FREQ'] = df[col].map(freq_map)
    df = df.drop(columns=[col])



In [74]:
#traitement bloc associé
bloc_associe = ['AGE1','AGE2','AGE3','ANTP1','ANTP2','ANTP3','ENGA1','ENGA2','ENGA3',
                'EXP1','EXP2','EXP3','LFAS1','LFAS2','LFAS3','LOCALE','NIVED1','NIVED2',
                'NIVED3','NOTE11','NOTE12','NOTE13','NOTE14','PATR1','PATR2','PATR3',
                'QALDE','RGI']

cols_associe_present = [c for c in bloc_associe if c in df.columns]

df['FLAG_PAS_ASSOCIE'] = df[cols_associe_present[0]].isna().astype(int)
df[cols_associe_present] = df[cols_associe_present].fillna(0)


In [75]:
#traitement bloc financier

bloc_financier = ['CU','CV','CW','DO','DU','ET','FF','FI','GI2','GI3','HA1','HA3','HA4',
                   'HA5','HDG','HP','HZ','JCX','KW','LP','LV','MC','MD','MP','MT','MU']
cols_financier_present = [c for c in bloc_financier if c in df.columns]

# Flag avant imputation
df['FLAG_DONNEES_FINANCIERES_MANQUANTES'] = df[cols_financier_present[0]].isna().astype(int)

# Imputation KNN 
knn_imputer = KNNImputer(n_neighbors=5)
scaler_knn = StandardScaler()
df_scaled = pd.DataFrame(scaler_knn.fit_transform(df[cols_financier_present]), 
                          columns=cols_financier_present, index=df.index)
df_scaled_imputed = pd.DataFrame(knn_imputer.fit_transform(df_scaled),
                                   columns=cols_financier_present, index=df.index)
df[cols_financier_present] = scaler_knn.inverse_transform(df_scaled_imputed)

df = df.copy()

In [76]:
df = df.dropna(subset=['TAUX_INTERET', 'PERIODICITE_MOIS'])
df = df.reset_index(drop=True)

In [77]:
for col in ['AGE_ENTREPRISE_ANS', 'VILLE_RC_FREQ', 'VILLE_TIERS_FREQ']:
    df[col] = df[col].fillna(df[col].median())


In [78]:
df.to_csv("df_part2.csv")

In [79]:

def winsoriser(df, cols, lower_pct=0.01, upper_pct=0.99):
    df_out = df.copy()
    bornes = {}
    for col in cols:
        low = df_out[col].quantile(lower_pct)
        high = df_out[col].quantile(upper_pct)
        df_out[col] = df_out[col].clip(lower=low, upper=high)
        bornes[col] = (low, high)
    return df_out, bornes


exclure = ['CODE_SECT_ACTIVITE', 'CODE_BRANCHE_ACTIVITE', 'CODE_SSBRANCHE_ACT',
           'ID_IFP', 'DEFAUT']
exclure += [c for c in df.columns if c.startswith('FLAG_')]
exclure += df.select_dtypes(include='bool').columns.tolist()  
exclure += [c for c in df.columns if df[c].dropna().between(0, 100).all() 
            and df[c].nunique() < 50]  

cols_a_winsoriser = [c for c in df.select_dtypes(include=['float64','int64']).columns 
                     if c not in exclure]


df, bornes_winsorisation = winsoriser(df, cols_a_winsoriser, lower_pct=0.01, upper_pct=0.99)
df = df.copy()

In [ ]:
def winsoriser(df, cols, lower_pct=0.01, upper_pct=0.99):
    df_out = df.copy()
    for col in cols:
        low, high = df_out[col].quantile(lower_pct), df_out[col].quantile(upper_pct)
        df_out[col] = df_out[col].clip(lower=low, upper=high)
    return df_out

cols_financieres_brutes = ['CU','CV','CW','DO','DU','ET','FF','FI','GI2','GI3','HA1',
                            'HA3','HA4','HA5','HDG','HP','HZ','JCX','KW','LP','LV',
                            'MC','MD','MP','MT','MU','RGI']

df = winsoriser(df, [c for c in cols_financieres_brutes if c in df.columns])

def safe_div(num, denom):
    return num / denom.replace(0, np.nan)

new_cols = {}
new_cols['RA1_FDR']                  = df['MC'] - df['CW']
new_cols['RA3_BFDR']                 = df['ET'] - df['MP']
new_cols['RA6_Tresorerie_nette']     = df['FF'] - df['MT']
new_cols['RA2_FDR_jours_CA']         = safe_div(new_cols['RA1_FDR'], df['RGI']) * 360
new_cols['RA4_BFDR_jours_CA']        = safe_div(new_cols['RA3_BFDR'], df['RGI']) * 360
new_cols['RA5_Couverture_BFDR_FDR']  = safe_div(new_cols['RA1_FDR'], new_cols['RA3_BFDR']) * 100
new_cols['RA7_TN_jours_CA']          = safe_div(new_cols['RA6_Tresorerie_nette'], df['RGI']) * 360
new_cols['RA8_CP_sur_CapPermanents'] = safe_div(df['LP'], df['MC']) * 100
new_cols['RA9_CP_sur_TotalBilan']    = safe_div(df['LP'], df['FI']) * 100
new_cols['RA10_DLMT_sur_CAF']        = safe_div(df['LV'], df['HA5']) * 100
new_cols['RA11_ChargesFin_sur_CA']   = safe_div(df['JCX'], df['RGI']) * 100
new_cols['RA12_ImmoNettes_sur_Bilan']= safe_div(df['CW'], df['FI']) * 100
new_cols['RA13_AmortCumul_sur_ImmoBrutes'] = safe_div(df['CV'], df['CU']) * 100
new_cols['RA14_BeneficeNet_sur_CA']  = safe_div(df['KW'], df['RGI']) * 100
new_cols['RA15_ResultatNet_sur_CP']  = safe_div(df['KW'], df['LP']) * 100
new_cols['RA16_CAF_sur_CA']          = safe_div(df['HA5'], df['RGI']) * 100
new_cols['RA17_DelaisCreancesClients'] = safe_div(df['DU'], df['RGI']) * 360
new_cols['RA18_DelaisCreditsFournisseurs'] = safe_div(df['MD'], df['HDG']) * 360
new_cols['RA19_Stocks_jours_CA']     = safe_div(df['DO'], df['RGI']) * 360
new_cols['RA20_BFDR_jours_CA']       = new_cols['RA4_BFDR_jours_CA']
new_cols['EVOL_CA_N_N1']             = safe_div(df['RGI'] - df['GI2'], df['GI2']) * 100
new_cols['EVOL_CA_N1_N2']            = safe_div(df['GI2'] - df['GI3'], df['GI3']) * 100

df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
ratio_cols = list(new_cols.keys())

flag_df = pd.DataFrame({f'FLAG_{c}_NC': df[c].isna().astype(int) for c in ratio_cols}, index=df.index)
df = pd.concat([df, flag_df], axis=1)


jours_cols = [c for c in ratio_cols if 'jours' in c.lower()]
pct_cols = [c for c in ratio_cols if c not in jours_cols and c not in 
            ['RA1_FDR','RA3_BFDR','RA6_Tresorerie_nette']]  

for col in jours_cols:
    df[col] = df[col].clip(lower=-1000, upper=1000)
for col in pct_cols:
    df[col] = df[col].clip(lower=-500, upper=500)

df = df.copy()

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

def evaluer_modeles(df, target_col='DEFAUT', n_splits=5, random_state=42):
    y = df[target_col].values
    X = df.drop(columns=[target_col]).astype(float)

    scale_pos_weight = (y == 0).sum() / (y == 1).sum()

    modeles = {
        'Régression Logistique': lambda: LogisticRegression(
            max_iter=2000, class_weight='balanced', random_state=random_state),
        'Random Forest': lambda: RandomForestClassifier(
            n_estimators=300, class_weight='balanced', random_state=random_state),
        'XGBoost': lambda: XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            scale_pos_weight=scale_pos_weight, eval_metric='logloss',
            random_state=random_state),
    }


    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scores = {name: [] for name in modeles}
    oof_proba = {name: np.zeros(len(y)) for name in modeles}

    for tr_idx, te_idx in skf.split(X, y):
        Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
        ytr, yte = y[tr_idx], y[te_idx]

       
        imp = SimpleImputer(strategy='median')
        Xtr_i = pd.DataFrame(imp.fit_transform(Xtr), columns=X.columns)
        Xte_i = pd.DataFrame(imp.transform(Xte), columns=X.columns)

 
        sc = StandardScaler()
        Xtr_s = sc.fit_transform(Xtr_i)
        Xte_s = sc.transform(Xte_i)

        for name, build_model in modeles.items():
            model = build_model()
            if name == 'Régression Logistique':
                model.fit(Xtr_s, ytr)
                proba = model.predict_proba(Xte_s)[:, 1]
            else:
                model.fit(Xtr_i, ytr)
                proba = model.predict_proba(Xte_i)[:, 1]

            pred = (proba >= 0.5).astype(int)
            oof_proba[name][te_idx] = proba
            scores[name].append([
                roc_auc_score(yte, proba),
                accuracy_score(yte, pred),
                precision_score(yte, pred, zero_division=0),
                recall_score(yte, pred, zero_division=0),
                f1_score(yte, pred, zero_division=0),
            ])


    lignes = []
    for name, vals in scores.items():
        arr = np.array(vals)
        m, s = arr.mean(axis=0), arr.std(axis=0)
        lignes.append({
            'Modèle': name,
            'AUC': f"{m[0]:.3f}±{s[0]:.3f}",
            'Accuracy': f"{m[1]:.3f}±{s[1]:.3f}",
            'Precision': f"{m[2]:.3f}±{s[2]:.3f}",
            'Recall': f"{m[3]:.3f}±{s[3]:.3f}",
            'F1': f"{m[4]:.3f}±{s[4]:.3f}",
        })
    resultats = pd.DataFrame(lignes)

    return resultats, oof_proba


resultats, oof_proba = evaluer_modeles(df)
print(resultats.to_string(index=False))

               Modèle         AUC    Accuracy   Precision      Recall          F1
Régression Logistique 0.688±0.060 0.693±0.042 0.328±0.054 0.602±0.097 0.423±0.061
        Random Forest 0.728±0.032 0.801±0.023 0.449±0.092 0.345±0.122 0.382±0.104
              XGBoost 0.718±0.041 0.788±0.020 0.410±0.085 0.378±0.113 0.392±0.102
